# 2-buses UCBlock, NetworkBlock and two ThermalUnitBlocks

In the following, we provide a quick example on how to add a simple optimization model with **SMS++**.
The problem below optimizes the dispatch of two thermal generators connected by a transmission line
between two buses over 24 hours.

In [ ]:
import numpy as np

from pysmspp import Block, SMSConfig, SMSFileType, SMSNetwork, Variable

sn = SMSNetwork(file_type=SMSFileType.eBlockFile)  # Empty Block

sn

## Creating an SMS++ Network

First, we create an empty SMS++ network with the block file format and import necessary components:

## Defining the UCBlock Parameters

The network does not contain any block inside, so it has to be populated. The first
step is to specify the main parameters of the UCBlock.

1- Parameters for time, number of units, generators, and nodes:

In [ ]:
kwargs = {
    "TimeHorizon": 24,  # number of time steps
    "NumberUnits": 2,  # number of units
    "NumberElectricalGenerators": 2,  # number of electrical generators
    "NumberNodes": 2,  # number of nodes
    "NumberLines": 1,  # number of lines
}

2- **Demand for each node**: This has to be defined as a Variable object:

In [ ]:
demand_array = np.full((2, 24), 50.0)
demand = {
    "ActivePowerDemand": Variable(  # active power demand
        "ActivePowerDemand",
        "float",
        ("NumberNodes", "TimeHorizon"),
        demand_array,
    )
}  # constant demand of 50kW

kwargs = {**kwargs, **demand}

3- **Parameters for the line**: A line can be described with a DCNetworkBlock (DC power flow) or a
TransportBlock. Here we use a TransportBlock with the following parameters:

In [ ]:
line_variables = {
    "StartLine": Variable("StartLine", "int", ("NumberLines",), [0]),
    "EndLine": Variable("EndLine", "int", ("NumberLines",), [1]),
    "MinPowerFlow": Variable("MinPowerFlow", "float", ("NumberLines",), [-50.0]),
    "MaxPowerFlow": Variable("MaxPowerFlow", "float", ("NumberLines",), [50.0]),
    "LineSusceptance": Variable("LineSusceptance", "float", ("NumberLines",), [0.0]),
}

kwargs = {**kwargs, **line_variables}

4- **Generator location**: Variable to specify in which bus (node) the generator is attached:

In [ ]:
generator_node = {
    "GeneratorNode": Variable(
        "GeneratorNode", int, ("NumberElectricalGenerators",), [0, 1]
    ),
}

kwargs = {**kwargs, **generator_node}

## Adding the UCBlock to the Network

Add everything with the SMSNetwork.add function:

In [ ]:
sn.add(
    "UCBlock",  # block type
    "Block_0",  # block name
    id="0",  # block id
    **kwargs,
)

sn

## Inspecting the Network Structure

Now the SMSNetwork object has a UCBlock called Block_0. Let's see how it is organized:

In [ ]:
sn.blocks["Block_0"]

## Adding Thermal Unit Blocks

Now, the two thermal units have to be added to the UCBlock as ThermalUnitBlocks.
First, we create a thermal unit block with the following parameters:

In [ ]:
thermal_unit_block = Block().from_kwargs(
    block_type="ThermalUnitBlock",
    MinPower=Variable("MinPower", "float", (), 0.0),
    MaxPower=Variable("MaxPower", "float", (), 70.0),
    LinearTerm=Variable("LinearTerm", "float", (), 0.3),
    InitUpDownTime=Variable("InitUpDownTime", "int", (), 1),
)

thermal_unit_block

Then, the unit block is added to the UCBlock:

In [ ]:
# Add it to the existing UCBlock (Block_0)
sn.blocks["Block_0"].add_block("UnitBlock_0", block=thermal_unit_block)

sn.blocks["Block_0"]

Similarly for the second ThermalUnitBlock. The max power is chosen to force the unit to
be turned on to supply the demand:

In [ ]:
thermal_unit_block = Block().from_kwargs(
    block_type="ThermalUnitBlock",
    MinPower=Variable("MinPower", "float", (), 0.0),
    MaxPower=Variable("MaxPower", "float", (), 90.0),
    LinearTerm=Variable("LinearTerm", "float", (), 0.8),
    InitUpDownTime=Variable("InitUpDownTime", "int", (), 1),
)

# Add it to the existing UCBlock (Block_0)
sn.blocks["Block_0"].add_block("UnitBlock_1", block=thermal_unit_block)

sn.blocks["Block_0"]

## Optimizing the Network

The problem can now be optimized using a solver configuration:

In [ ]:
configfile = SMSConfig(
    template="UCBlock/uc_solverconfig"
)  # path to the template solver config file "uc_solverconfig"
temporary_smspp_file = "./2buses_2thermal.nc"  # path to temporary SMS++ file
output_file = "./2buses_2thermal.txt"  # path to the output file (optional)
fp_solution = "./fp_solution_2buses_2thermal.nc4"  # path to the file where the full problem solution will be saved (optional). When provided, the result.solution object will be populated with the SMS++ solution object

result = sn.optimize(
    configfile,
    temporary_smspp_file,
    output_file,
    fp_solution=fp_solution,
)

## Viewing Results

The value of the objective function and the complete log can be obtained with:

In [ ]:
result.objective_value

In [ ]:
result.log

In [ ]:
result.solution